In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv("IMDB Dataset.csv")

In [16]:
#handle duplcates
df.drop_duplicates(inplace = True)

  #  Text Preprocessing


 ### 1. converting to lower case

In [42]:
df["review"] = df["review"].str.lower()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive


  ### 2. Removing the URL

In [20]:
#Regular expression method use 
import re
def remove_urls(text):
    text = re.sub(r"http\S+", "",text) #pattern,replace,string
    return text

df["review"] = df["review"].apply(remove_urls)    


  ### 3 . Remove Punctuation

In [21]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "",text) # A-Z, a-z,0-9
    return text
df["review"] = df["review"].apply(remove_punctuations)     

 ### 4. Remove HTML

In [22]:
def remove_html(text):
    text = re.sub(r"<,*?>", "",text)
    return text
df["review"] = df["review"].apply(remove_html)   

### 5 .  Removing stopwords using NLTK

In [27]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [25]:
def remove_stopword(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text        
            

 ### 6 . Stemming

In [ ]:
#reducing a word to its root (stem) by removing prefixes or suffixes.
#running -> run
#played->play
#porterstemming -> Porter Stemmer is a stemming algorithm 
#provided by the NLTK library in Python

In [35]:
from nltk.stem import PorterStemmer
def stemming(text):
    ps = PorterStemmer()
    stemmed_word = []

    
    tokens = word_tokenize(text)
    for word in tokens:
        stemmed_tokens = ps.stem(word)
        stemmed_word.append(stemmed_tokens)
    return " ".join(stemmed_word)  

df["review"] = df["review"].apply(stemming)       

 7. # Encoding

In [45]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

Y= df["sentiment"]

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])


In [52]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(
    X,Y,test_size = 0.2,random_state = 42 
)

In [58]:
#sparse matrix convert into numpy array
X_train = X_train.toarray()
X_test = X_test.toarray()

In [60]:
import torch
from torch.utils.data import TensorDataset,DataLoader
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [61]:
train_loader = DataLoader(train_set,batch_size = 64, shuffle = True)
test_loader = DataLoader(test_set,batch_size = 64, shuffle = True)

 # Build our RNN

In [74]:
import torch.nn as nn
import torch.optim as optim

In [75]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size = 128, num_layers = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        ##RNN Layers
        self.rnn = nn.RNN(input_size,hidden_size,num_layers,batch_first = True)
        ## fully connected layers
        self.fc = nn.Linear(hidden_size,1)
    #forwardm propagation
    def forward(self,x):
        #optional
        h0 = torch.zeros(self.num_layers,x.size(0),self.hidden_size)
        out, _ = self.rnn(x,h0)
        #1st values = hidden states of all the timestamp => (batch_size,seq_len,hidden_size)
        #2nd value = final hidden state of last timestamp

        out = self.fc(out[:,-1,:])
        return out
        

In [76]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

 ## Training The RNN

In [78]:
epochs = 10
for epoch in range(epochs):
    model.train()
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        Xb = Xb.unsqueeze(1)
        outputs = model (Xb)

        outputs = torch.sigmoid(outputs.squeeze())

        loss = criterion(outputs,yb) #compute loss
        loss.backward()
        optimizer.step() #weight update
    print(f"epochs = {epoch+1}/{epochs} and loss = {loss.item()}")        

epochs = 1/10 and loss = 0.24882280826568604
epochs = 2/10 and loss = 0.21423326432704926
epochs = 3/10 and loss = 0.35369935631752014
epochs = 4/10 and loss = 0.1270989179611206
epochs = 5/10 and loss = 0.1705441027879715
epochs = 6/10 and loss = 0.24163717031478882
epochs = 7/10 and loss = 0.1725219488143921
epochs = 8/10 and loss = 0.15913939476013184
epochs = 9/10 and loss = 0.27943849563598633
epochs = 10/10 and loss = 0.18897965550422668


In [81]:
# Evaluation
model.eval()
with torch.no_grad():
    correct_val = 0
    total_val  = 0
    for Xb,yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        total_val += yb.size(0)
        correct_val += (predicted == yb).sum().item()
    print(f"accuracy =  {correct_val/total_val*100}")   
    

accuracy =  87.15337299586568


In [83]:
model.eval()

review = input("Enter Review: ")

# Preprocessing
review = stemming(review)

# TF-IDF
review = tf.transform([review])

# Tensor
review = torch.tensor(review.toarray(), dtype=torch.float32)

# RNN ke liye sequence dimension
review = review.unsqueeze(1)

with torch.no_grad():
    output = model(review)
    prediction = (torch.sigmoid(output.squeeze()) > 0.5).float()

    if prediction.item() == 1:
        print("😊 Positive Review")
    else:
        print("😞 Negative Review")

Enter Review:  movie is bad


😞 Negative Review


In [84]:
#saving model
torch.save(model.state_dict(), "sentiment_model.pth")

In [85]:
import pickle

with open("tfidf.pkl", "wb") as f:
    pickle.dump(tf, f)

In [86]:
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)